# Phase 3B: frozen full-development baseline
Run one whole dataset per batch in the documented order. Default 01 is a NEW ETs execution, not reuse of the smoke score. 1,563 development images; no confirmation or hidden-test inference. Labels are used only by the separate post-prediction evaluator. Read docs/step03b_supervisor_handoff.md before execution.

In [ ]:
import base64, hashlib, io, json, sys, zipfile
from pathlib import Path
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='6b3ddf1278a9786af2dd3b5ee21fb395ea0a21c150ad2fb35ae1980f408fb8dd'
ROOT=Path('/kaggle/working/phase3b_source_d5d612c530d5')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target=(ROOT/name).resolve()
        assert target.is_relative_to(ROOT.resolve())
        data=z.read(name)
        if target.exists() and target.read_bytes()!=data:raise RuntimeError('Source differs; use a fresh session')
        target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists():target.write_bytes(data)
print('Phase 3B source verified:', 'd5d612c530d52089b6dfa7ab3589ccaee572d0ba6281b9bfb35ceba044e5ce93')


In [ ]:
# Edit only these session controls. Model settings and split are immutable.
# Use a fork pinned to the original Python 3.10.12 environment, T4, Internet OFF.
BATCH_IDS = ['01']  # 01..11 in docs/step03b_execution_plan.md
SESSION_REMAINING_SECONDS = 3600  # actual time remaining, <=21600; reserves 600 s
EXPORT_CACHE = True  # optional features/database cache; no source images/weights
RESUME_ARCHIVES = []  # absolute mounted report AND cache .zip paths for same dataset
EXPORT_ID = '_'.join(BATCH_IDS)
WORK = Path('/kaggle/working/phase3b_'+EXPORT_ID)
def mounted(*candidates):
    found = [p for p in map(Path, candidates) if p.exists()]
    # Keep the expected path if missing so Stage A records a useful failure archive.
    if len(found) > 1: raise RuntimeError('Ambiguous mounted assets; specify one exact input')
    return str(found[0] if found else Path(candidates[0]))
mounts = {
 'competition': mounted('/kaggle/input/competitions/image-matching-challenge-2025-ongoing', '/kaggle/input/image-matching-challenge-2025-ongoing'),
 'packages': mounted('/kaggle/input/imc2024-packages-lightglue-rerun-kornia', '/kaggle/input/datasets/oldufo/imc2024-packages-lightglue-rerun-kornia'),
 'dinov2': mounted('/kaggle/input/dinov2/pytorch/base/1', '/kaggle/input/models/metaresearch/dinov2/pytorch/base/1'),
 'aliked': mounted('/kaggle/input/aliked/pytorch/aliked-n16/1/aliked-n16.pth', '/kaggle/input/models/oldufo/aliked/pytorch/aliked-n16/1/aliked-n16.pth'),
 'lightglue': mounted('/kaggle/input/lightglue/pytorch/aliked/1/aliked_lightglue.pth', '/kaggle/input/models/oldufo/lightglue/pytorch/aliked/1/aliked_lightglue.pth')}

settings={'work':str(WORK),'batches':BATCH_IDS,'export_id':EXPORT_ID,
 'session_remaining_seconds':SESSION_REMAINING_SECONDS,'export_cache':EXPORT_CACHE,
 'resume_archives':RESUME_ARCHIVES,'mounts':mounts}
SETTINGS=Path('/kaggle/working/phase3b_settings.json')
SETTINGS.write_text(json.dumps(settings))
print('Development only; batches:', BATCH_IDS, 'remaining seconds:',SESSION_REMAINING_SECONDS)


In [ ]:
import subprocess
result=subprocess.run([sys.executable,str(ROOT/'src/baselines/development/runner.py'),'--settings',str(SETTINGS)])
print('Runner exit:',result.returncode)
print('Download all imc_baseline_dev_batch_*.zip files from /kaggle/working, including partial reports and cache archives.')
print('Return archives for independent validation. No full baseline is registered by this notebook.')
if result.returncode:print('STOP: inspect failure receipt. Preserve model settings; resume only verified checkpoints.')
